# 1. 문제 정의
- 타이타닉 데이터를 이용해서 생존/사망을 예측해보자.
- 다양한 전처리 과정을 익혀보자.
- 랜덤포레스트 모델을 이용하여 분석하자.

# 2. 데이터 수집
- 'titanic.csv' 불러와서 tatanic 변수에 담기

In [1]:
# 도구 불러오기
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [5]:
# csv 파일 titanic 변수에 담기
# pd.read_csv(경로/이름, index_col=인덱스할 컬럼이름)
titanic = pd.read_csv('./data/titanic.csv', index_col = 'PassengerId')
titanic.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 컬럼 내용
- 'Survived' : 생존/사망 결과 값이 담긴 컬럼(0:사망, 1:생존)
- 'Pclass' : 객실의 등급(1:상위층, 2:중상위층, 3:하위층)
- 'Name'
- 'Sex'
- 'Age'
- 'SibSp' : 형제자매/배우자 수 (같이 탑승한)
- 'Parch' : 부모/자식의 수(같이 탑승한)
- 'Ticket' : 티켓의 번호
- 'Fare' : 요금
- 'Cabin' : 객실 번호
- 'Embarked' : 승선항 (S, C, Q)
- S = Southampton, C = Cherbourg, Q = Queenstown

# 3. 전처리 & 4. EDA

In [7]:
# 데이터 정보 확인(info())
titanic.info()

# 'Age', 'Cabin', 'Embarked' 에서 결측치 확인
# 숫자형, 문자형 확인 (인코딩 필요)

<class 'pandas.core.frame.DataFrame'>
Index: 891 entries, 1 to 891
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Name      891 non-null    object 
 3   Sex       891 non-null    object 
 4   Age       714 non-null    float64
 5   SibSp     891 non-null    int64  
 6   Parch     891 non-null    int64  
 7   Ticket    891 non-null    object 
 8   Fare      891 non-null    float64
 9   Cabin     204 non-null    object 
 10  Embarked  889 non-null    object 
dtypes: float64(2), int64(4), object(5)
memory usage: 83.5+ KB


## 결측치 채우기
1. 결측치가 적을 때 삭제하는 방법
    - 데이터 자체가 귀하기 때문에 추천X
2. 결측치를 다른 대푯값(평균값, 중간값 등)으로 대체하는 방법

In [8]:
# 결측치 확인
titanic.isnull().sum()

Survived      0
Pclass        0
Name          0
Sex           0
Age         177
SibSp         0
Parch         0
Ticket        0
Fare          0
Cabin       687
Embarked      2
dtype: int64

### (1) titanic['Embarked'] 결측치 채우기

In [10]:
# Embarked(승선항) 데이터 개수 확인
titanic['Embarked'].value_counts()

Embarked
S    644
C    168
Q     77
Name: count, dtype: int64

In [11]:
titanic['Embarked'].unique()

array(['S', 'C', 'Q', nan], dtype=object)

In [13]:
# Embarked에서 결측치가 있는 데이터를 추출하자
# 불리언인덱싱으로 가져오기 위해서 조건식 세우기
# titanic['Embarked'].isnull() -> 조건식

titanic[titanic['Embarked'].isnull()]
# 두사람이 동승을 했을 것으로 예상

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
62,1,1,"Icard, Miss. Amelie",female,38.0,0,0,113572,80.0,B28,NaN
830,1,1,"Stone, Mrs. George Nelson (Martha Evelyn)",female,62.0,0,0,113572,80.0,B28,NaN


In [14]:
# Pclass, Embarked 그룹화
# groupby('기준컬럼')[계산컬럼].계산함수()
# pclass를 기준으로 embarked의 개수를 확인
titanic.groupby('Pclass')['Embarked'].value_counts()

Pclass  Embarked
1       S           127
        C            85
        Q             2
2       S           164
        C            17
        Q             3
3       S           353
        Q            72
        C            66
Name: count, dtype: int64

In [16]:
# 1등급인 사람들이 'S' 승선항에서 제일 많이 승선
# 결측치 데이터가 1등급인 사람들 -> 결측치를 'S'로 채워주자(fillna(채울값))
titanic['Embarked'] = titanic['Embarked'].fillna('S')

In [17]:
# 결측치 확인
titanic['Embarked'].isnull().sum()

np.int64(0)

## (2) 'Age' 결측치

In [19]:
# 'Age'가 결측치인 데이터를 추출 (불리언인덱싱)
# 조건식 titanic['Age'].isnull()
titanic[titanic['Age'].isnull()]

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
18,1,2,"Williams, Mr. Charles Eugene",male,NaN,0,0,244373,13.0000,NaN,S
20,1,3,"Masselmani, Mrs. Fatima",female,NaN,0,0,2649,7.2250,NaN,C
27,0,3,"Emir, Mr. Farred Chehab",male,NaN,0,0,2631,7.2250,NaN,C
29,1,3,"O'Dwyer, Miss. Ellen ""Nellie""",female,NaN,0,0,330959,7.8792,NaN,Q
...,...,...,...,...,...,...,...,...,...,...,...
860,0,3,"Razi, Mr. Raihed",male,NaN,0,0,2629,7.2292,NaN,C
864,0,3,"Sage, Miss. Dorothy Edith ""Dolly""",female,NaN,8,2,CA. 2343,69.5500,NaN,S
869,0,3,"van Melkebeke, Mr. Philemon",male,NaN,0,0,345777,9.5000,NaN,S


In [20]:
# 데이터의 상관관계 확인
# corr() -> 문자형 데이터가 들어가면 에러
titanic.corr(numeric_only = True)
# Pclass 가 Age에 가장 큰 상관관계가 있다는 것을 확인(-0.369226)

,Survived,Pclass,Age,SibSp,Parch,Fare
Survived,1.000000,-0.338481,-0.077221,-0.035322,0.081629,0.257307
Pclass,-0.338481,1.000000,-0.369226,0.083081,0.018443,-0.549500
Age,-0.077221,-0.369226,1.000000,-0.308247,-0.189119,0.096067
SibSp,-0.035322,0.083081,-0.308247,1.000000,0.414838,0.159651
Parch,0.081629,0.018443,-0.189119,0.414838,1.000000,0.216225
Fare,0.257307,-0.549500,0.096067,0.159651,0.216225,1.000000


In [24]:
# Pclass, Sex, Age를 그룹화해서 Age의 평균값을 구하기(groupby)
# 데이터.groupby('기준컬럼')['계산하고싶은컬럼'].계산함수()
ps_gb = titanic.groupby(['Pclass','Sex'])['Age'].mean().astype('int')
ps_gb

Pclass  Sex   
1       female    34
        male      41
2       female    28
        male      30
3       female    21
        male      26
Name: Age, dtype: int64

In [25]:
# 3클래스의 남성 평균 나이값(26)을 가져오기
# data.loc[행]
ps_gb.loc[(3,'male')]

np.int64(26)

In [28]:
# 인덱스 6인 데이터 가져오기
titanic.loc[6, 'Age'] 

# 가져온 값에 3클래스 남성의 평균나이값 대입
titanic.loc[6, 'Age'] = ps_gb.loc[(3,'male')]

In [29]:
titanic.loc[6, 'Age'] 

np.float64(26.0)

In [41]:
# 인덱스 18인 데이터 가져오기(2클래스, 남성)
titanic.loc[18,:]

Survived                               1
Pclass                                 2
Name        Williams, Mr. Charles Eugene
Sex                                 male
Age                                 30.0
SibSp                                  0
Parch                                  0
Ticket                            244373
Fare                                13.0
Cabin                                NaN
Embarked                               S
Name: 18, dtype: object

In [42]:
# 가져온 18번 인덱스 Age값에 (2)클래스인 (male)의 평균나이값을 대입
titanic.loc[18,'Age'] = ps_gb.loc[(2,'male')]

In [40]:
# 인덱스 20인 데이터 가져오기
titanic.loc[20,:]

Survived                          1
Pclass                            3
Name        Masselmani, Mrs. Fatima
Sex                          female
Age                            21.0
SibSp                             0
Parch                             0
Ticket                         2649
Fare                          7.225
Cabin                           NaN
Embarked                          C
Name: 20, dtype: object

In [44]:
# 가져온 인덱스 20의 Age값에 인덱스 20의 Pclass 이면서 인덱스 20의 Sex의 평균나이값(ps_gb) 대입
titanic.loc[20,'Age'] = ps_gb.loc[(titanic.loc[20,'Pclass'],titanic.loc[20,'Sex'])]

In [45]:
# 확인
titanic.loc[20,'Age']

np.float64(21.0)

In [ ]:
# isnull() => pandas 함수
# isnan() => numpy 함수

In [47]:
# 반복문으로 모든 인덱스에 대해서 결측치가 맞다면 클래스별성별별 평균나이값(ps_gb) 대입
for i in titanic.index : 
    # 성별 컬럼이 결측치인지 조건문을 추가
    if np.isnan(titanic.loc[i, 'Age']) : 
        # 반복해서 작업하고 싶은 결측치 채우는 작업
        titanic.loc[i,'Age'] = ps_gb.loc[(titanic.loc[i,'Pclass'],titanic.loc[i,'Sex'])]

In [48]:
# 결측치 확인
titanic['Age'].isnull().sum()

np.int64(0)